# So far:

- ASCII encoding (mostly, theres a big glaring bug)

- prime generation (mostly, theres also a bug here)

- other RSA functions

- easy-ish imports (will this work in CoLab?)

- rough sizes of numbers known

# To Do:

- implement breaking up longer messages into shorter words

- put it all together at least once

8 characters--> 64 bits --> 33 bit primes --> primes between 2\**33 and 2\**34

In [1]:
import project_path
from rsatools import *

In [ ]:
rng = prime_rng(8675_309)

p = rng.randprime(2**31, 2**32)
q = rng.randprime(2**31, 2**32)

n = p*q

lamb = carmichael(p, q)

e = 257
d = inverse257(lamb)

secrets = [p, q, n, lamb, e, d]

for num in secrets:
    bitlength = len(str(bin(num)))-2
    print(f"{bitlength:2.0f}    {num}")

32    3721615301
32    2961235187
64    11020578181798796287
59    423868391350613300
 9    257
58    222654602460438905


clearly the bitlength calculations I did earlier was wrong...

In [3]:
"""
m_plaintext = "Hello, W"

m_encoding = int(codecs.encode(m_plaintext, encoding="ascii_emoji").hex(), 16)

c_encoding = (m_encoding ** e) % n

c_plaintext = codecs.decode(bytes.fromhex(hex(c_encoding)[2:]), encoding="ascii_emoji")

print(m_plaintext+"  -->  "+c_plaintext)"""
pass

I've tried to decode this fucking thing for 5 minutes and jupyter hasn't completed.  So we're gonna need to implement fast modular exponentiation by hand.  fuck.

In [4]:
"""r_encoding = (c_encoding ** d) % n

r_plaintext = codecs.decode(bytes.fromhex(hex(r_encoding)[2:]), encoding="ascii_emoji")"""
pass

Yoni coming in clutch by informing me that pow already has a mod parameter.  I probably should have checked that before just immediately implementing it myself.  

In [5]:
m_plaintext = "Hello, W"

m_encoding = int(codecs.encode(m_plaintext, encoding="ascii_emoji").hex(), 16)

c_encoding = pow(m_encoding, e, mod=n)

c_plaintext = codecs.decode(bytes.fromhex(hex(c_encoding)[2:]), encoding="ascii_emoji")

r_encoding = pow(c_encoding, d, mod=n)

print(bytes.fromhex(str(hex(m_encoding))[2:]))
print(bytes.fromhex(str(hex(c_encoding))[2:]))
print(bytes.fromhex(str(hex(r_encoding))[2:]))
#print(m_plaintext+"  -->  "+c_plaintext)

b'Hello, W'
b'(a\x14lKhW3'
b'[\x1c\xb2s\x93\xac\xb7\x1c'


Initial Output of above code block saved for history:

```
b'Hello, W'
b'(a\x14lKhW3'
b'K\xdb\x88\t\xbc\xeb\xc7t'
```

FUUUUUUUUUUUUUUUUUUUUUUUUUUCK

okay I realized that d is supposed to be the inverse of e mod lambda, not mod n.  so I changed that and the shit still doesn't work.  

```
b'Hello, W'
b'(a\x14lKhW3'
b'[\x1c\xb2s\x93\xac\xb7\x1c'
```

In [6]:
print((d*e)%lamb)

3085


well- that's not right.  

also wait lambda is also wrong!

WOAH we've got a TON of errors in this bitch.

what happened?

In [ ]:
from math import lcm
print(lcm(p-1, q-1))
print(lcm(3_721_615_300, 2_961_235_186))

423868391350613300
423868391350613300


but the calculator online says 423,868,391,350,613,312.  which is distinctly a different number (although its very close.)

well:

$p-1=3\,721\,615\,300=2\times2\times5\times5\times13\times947\times3023$

$q-1=2\,961\,235\,186=2\times7\times13\times53\times306991$

In [10]:
3_721_615_300*2_961_235_186

11020578175115945800

In [12]:
11020578175115945800//13

847736782701226600

wait I'm sorry I'm so fucking lost.  what the fuck

# This is exactly the reason I insisted on having a seed to work off of.  This is the exact issue I was worried about.  My stubbornness pays off again